# 평가셋(Evaluation DataSet) 구축

### 환경 설정

In [24]:
import os
import sqlite3
import pandas as pd
import chromadb

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain.agents import create_agent
from pathlib import Path
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer


In [3]:
load_dotenv('.env')
load_dotenv('/.env')

if not os.getenv('OPENAI_API_KEY') :
    raise RuntimeError(
        'OpenAI API Key를 확인해 보세요.'
    )
else :
    model = ChatOpenAI(model = 'gpt-4o-mini', temperature=1, timeout=60)
    print('OpenAI API 연결 & MODEL 생성 완료')

OpenAI API 연결 & MODEL 생성 완료


### VectorDB (ChromaDB) 연결 / 임베딩 모델 생성

In [26]:
DB_PATH = Path("../chroma_db")
COLLECTION_NAME = "maplestory_guides"

#_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True, isolation_level=None, check_same_thread=False)
EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
client = chromadb.PersistentClient(path=str(DB_PATH))
collection = client.get_collection(name=COLLECTION_NAME)

print('데이터베이스 연결 및 임베딩 모델 생성 완료')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10023.01it/s]


데이터베이스 연결 및 임베딩 모델 생성 완료


### 평가셋 TEST

#### 평가셋 데이터 가져오기

In [29]:
# [rag_eval_questions_100.csv](/C:/Users/Playdata/Desktop/mle-01-p1-team3/docs/rag_eval_questions_100.csv)

evalset = pd.read_csv('../docs/rag_eval_gold_100.csv')
print('페이지 수 : ', len(evalset))

display(evalset.head(3))


페이지 수 :  100


,eval_id,persona,category,question,question_variant,source_type,difficulty,answerable,gold_source_refs,gold_source_ids,gold_source_ref_ids,gold_source_urls,reference_answer,must_include,nice_to_include,must_not_include,scoring_focus
0,E001,초보,게임시작,메이플스토리를 처음 시작하려면 무엇부터 해야 하나요?,처음 설치한 뒤 어떤 순서로 시작하면 되나요?,guide,easy,yes,게임 시작,chunk_0; chunk_1; chunk_2; chunk_3; chunk_4; c...,article:272,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"
1,E002,초보,게임시작,메이플스토리는 어떻게 설치하고 실행하나요?,게임을 시작하려면 설치 절차가 어떻게 되나요?,guide,easy,yes,게임 시작,chunk_0; chunk_1; chunk_2; chunk_3; chunk_4; c...,article:272,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"
2,E003,초보,게임시작,처음 시작할 때 선택할 수 있는 직업은 어떤 종류가 있나요?,초반에 어떤 직업군을 고를 수 있나요?,guide,easy,yes,메이플스토리 신규 · 복귀 용사 가이드,chunk_1378; chunk_1379; chunk_1380; chunk_1381...,article:147377,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"


##### 함수 정의

In [57]:
def search_ids(question: str, k: int = 3):
    """질문과 가장 가까운 조각 k개의 id를 순위 순서로 돌려준다."""
    query_embedding = embed_model.encode(
        [question],
        normalize_embeddings = True
    )

    result = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results = k,
        include = []
    )

    return result["ids"][0]

In [69]:
# 평가셋(evalset)구성
# 'eval_id', 'persona', 'category', 'question', 
# 'question_variant', 'source_type', 'difficulty', 
# 'answerable', 'gold_source_refs', 'gold_source_ids', 
# 'gold_source_ref_ids', 'gold_source_urls', 'reference_answer', 
# 'must_include', 'nice_to_include', 'must_not_include', 
# 'scoring_focus'

# 가장 무난한 질문 선택 (또는 평가셋 번호, 예: 'E001' ~ 'E100')
#credit_query = '초보에게 무난한 직업을 고를 때 어떤 기준으로 보면 좋을까요?'
credit_query = 'E003'

top3 = search_ids(credit_query)

for rank, chunk_id in enumerate(top3, 1) :
    print(f'{rank}위 {chunk_id}')

doc_rank = []
for chunk_id in top3 :
    doc_id = chunk_id.split(',')[0]
    if doc_id not in doc_rank :
        doc_rank.append(doc_id)

print('문서 순위 : ', doc_rank)

# 평가셋 번호로 추출
credit_row = evalset[evalset['eval_id'] == credit_query].iloc[0]

#credit_row = evalset[evalset['question'] == credit_query].iloc[0]
credit_gold = credit_row['gold_source_ids'].split(';')

credit_gold_docs = {c.split(',')[0] for c in credit_gold}
print(f'평가셋 문항 {credit_row['eval_id']}의 정답 조각 : {credit_gold}') #\n(문서: {credit_gold_docs})

# print('청킹 단위 평가 : ', bool(set(top3) & set(credit_gold)))
# print('문서 단위 평가 : ', bool(set(doc_rank) & credit_gold_docs))

print('=' * 100)

hit_k = int(bool(set(top3) & set(credit_gold)))
recall_k = len(set(top3) & set(credit_gold)) / len(set(credit_gold))
precision_k = len(set(top3) & set(credit_gold)) / 3

print('hit@k 평가 : ', hit_k)
print('RECALL@k 평가 : ', recall_k)
print('Precision@K 평가 : ', precision_k)

1위 chunk_395
2위 chunk_265
3위 chunk_413
문서 순위 :  ['chunk_395', 'chunk_265', 'chunk_413']
평가셋 문항 E003의 정답 조각 : ['chunk_1378', ' chunk_1379', ' chunk_1380', ' chunk_1381', ' chunk_1382', ' chunk_1383', ' chunk_1384', ' chunk_1385', ' chunk_1386', ' chunk_1387', ' chunk_1388', ' chunk_1389', ' chunk_1390', ' chunk_1391', ' chunk_1392', ' chunk_1393', ' chunk_1394', ' chunk_1395', ' chunk_1396', ' chunk_1397', ' chunk_1398', ' chunk_1399', ' chunk_1400', ' chunk_1401', ' chunk_1402', ' chunk_1403', ' chunk_1404', ' chunk_1405', ' chunk_1406', ' chunk_1407', ' chunk_1408', ' chunk_1409', ' chunk_1410', ' chunk_1411', ' chunk_1412', ' chunk_1413', ' chunk_1414', ' chunk_1415', ' chunk_1416', ' chunk_1417', ' chunk_1418', ' chunk_1419', ' chunk_1420', ' chunk_1421', ' chunk_1422', ' chunk_1423', ' chunk_1424', ' chunk_1425', ' chunk_1426', ' chunk_1427', ' chunk_1428', ' chunk_1429', ' chunk_1430', ' chunk_1431', ' chunk_1432', ' chunk_1433', ' chunk_1434', ' chunk_1435', ' chunk_1436', ' chun

In [65]:
for rank in doc_rank :
    result = collection.get(ids=rank, include=["documents", "metadatas"])
    print(result["documents"][0])
    print(result["metadatas"][0])
    print()

for gold in credit_gold :
    result = collection.get(ids=gold, include=["documents", "metadatas"])
    print(result["documents"][0])
    print(result["metadatas"][0])
    print()


엔하임 디펜스 에스페라의 프로텍트 에스페라 스페셜 컨텐츠는 해당 지역의 스토리 퀘스트를 완료해야 진행 가능 스토리 퀘스트를 완료하면 길라잡이 F11 를 통해 간편하게 이동 가능 에르다 스펙트럼 소멸의 여로 근방의 에르다를 관찰하고 기록하는 조사원 니나가 부상을 입게 되어 대신 조사를 도와주세요 소멸의 여로 스토리 퀘스트를 완료한 200레벨 이상의 캐릭터가
{'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/465', 'name': '아케인리버 스페셜 컨텐츠', 'article_id': 465, 'section_title': '스페셜 컨텐츠', 'board_id': 429467341, 'chunk_index': 715, 'source': 'guide'}

제한시간이 모두 소모되면 미션은 실패합니다 2 슈멧의 특제 포탑 사용 방법 슈멧의 특제 포탑은 앉기 기본 단축키 X 버튼을 눌러 사용 가능 포탄 발사 각도 조절 위 아래 방향 키 포탄 발사 스페이스 바 포탄 변경 A 에르다 포탄 S 슈퍼 에르다 포탄 카메라 이동 Shift 방향키 슈멧 특제 포탑에서 사용할 수 있는 포탄은 2종류이며 종류에 따라 소모되는
{'article_id': 465, 'source': 'guide', 'name': '아케인리버 스페셜 컨텐츠', 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/465', 'section_title': '스페셜 컨텐츠', 'chunk_index': 748, 'board_id': 429467341}

챔피언 레이드 드래곤 아일랜드에 도전 가능 등록된 챔피언 캐릭터로 유니온 UI의 챔피언 전장 버튼을 눌러 챔피언 레이드에 참여할 수 있음 2 진행 방법 챔피언 레이드 드래곤 아일랜드는 총 6개의 스테이지로 구성 현재 진행중인 스테이지 단계와 획득한 누적 점수 해당 스테이지에서 수행 필요한 미션 스테이지 진입 시 

IndexError: list index out of range